# 04 · Fine-tuning pretrained Transformers

Notebooks 02 and 03 learn Kiswahili almost entirely from 1,810 labelled tweets. Pretrained Transformer encoders instead start from a language model trained on millions of sentences. We add a classification head and **fine-tune** the whole network on our task (Devlin et al., 2019).

**Research question:** does pretraining on African languages beat general multilingual pretraining for Kiswahili sentiment?

| ID | Model | Params | Pretraining data |
|---|---|---|---|
| T1 | mBERT (`bert-base-multilingual-cased`) | 178M | Wikipedia in 104 languages, including Kiswahili (Devlin et al., 2019) |
| T2 | XLM-R base | 278M | CommonCrawl (CC-100) in 100 languages, including Kiswahili (Conneau et al., 2020) |
| T3 | AfriBERTa large | 126M | Trained **from scratch** on 11 African languages, including Kiswahili (Ogueji et al., 2021) |
| T4 | AfroXLMR base | 278M | XLM-R **further pretrained** on 17 African languages, including Kiswahili (MAFT; Alabi et al., 2022) |

Then, for the best model:
- **T5:** no class weighting (ablation)
- **T6:** frozen encoder, training only the classification head. This tests whether fine-tuning the encoder matters or whether pretrained features alone are enough.
- **T7** (optional, slower): AfroXLMR **large** (560M parameters)

**How the classifier works.** The tweet is split into subword tokens, and `<s>`/`[CLS]` is prepended. The encoder produces a contextual vector for every token. The vector of the first token is passed through a small classification head (dense → tanh → linear) to produce 3 logits. Fine-tuning updates all weights with a weighted cross-entropy loss.

> **Runtime:** use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). T1–T6 with 3 seeds each take roughly 25–40 minutes on a T4.

### Before you press *Run all*: get a Hugging Face token (2 minutes, one time only)

At the end, this notebook **automatically** does two things:
1. uploads the best model to your Hugging Face account
2. uploads all results (tables, predictions, figures) next to it

For that it needs a **write token**:
1. Sign up or log in at https://huggingface.co.
2. Open https://huggingface.co/settings/tokens and click **Create new token**.
3. Choose the **Write** token type, name it e.g. `colab`, and click **Create token**.
4. Copy the token (it starts with `hf_`).

When you *Run all*, the second code cell shows a box asking for the token. Paste it and press **Enter**. Then you can leave the notebook running.

In [ ]:
# --- Setup: works both locally and in Google Colab ---
import os, sys
REPO_URL = "https://github.com/Joellate/kiswahili-sentiment-analysis.git"

if "google.colab" in sys.modules:
    if not os.path.exists("kiswahili-sentiment-analysis"):
        !git clone -q $REPO_URL
    %cd kiswahili-sentiment-analysis
    %pip install -q -r requirements.txt
else:
    # running locally from notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, "src")
print("Working directory:", os.getcwd())

In [ ]:
# --- Hugging Face login: paste your WRITE token in the box below and press Enter ---
UPLOAD_TO_HUB = os.environ.get("SMOKE_TEST") != "1"
if UPLOAD_TO_HUB:
    from getpass import getpass
    from huggingface_hub import login, whoami
    login(token=getpass("Paste your Hugging Face WRITE token and press Enter: ").strip())
    HF_USER = whoami()["name"]
    print("Logged in to Hugging Face as", HF_USER, "- you can now leave the notebook running")

In [ ]:
import gc
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

from data import load_all, LABELS, ID2LABEL, LABEL2ID
from evaluate import aggregate_seeds, compute_metrics as score, log_experiment, plot_confusion, report
from preprocessing import clean_text

SMOKE = os.environ.get("SMOKE_TEST") == "1"   # tiny CPU run used only to check the code end-to-end
RUN_LARGE = False                             # set True to also fine-tune AfroXLMR-large (T7), ~3x slower
SEEDS = [13] if SMOKE else [13, 42, 2026]
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

MODELS = {
    "T1 mBERT": "google-bert/bert-base-multilingual-cased",
    "T2 XLM-R base": "FacebookAI/xlm-roberta-base",
    "T3 AfriBERTa large": "castorini/afriberta_large",
    "T4 AfroXLMR base": "Davlan/afro-xlmr-base",
}
if SMOKE:
    MODELS = {"T0 smoke AfroXLMR mini": "Davlan/afro-xlmr-mini"}

splits = load_all()
# Transformers have their own subword tokenisers, so we only remove noise.
# We keep case and numbers, because the pretrained vocabularies are cased.
prep = lambda t: clean_text(t, lowercase=False, replace_numbers=False)
texts = {s: [prep(t) for t in d.text] for s, d in splits.items()}
y = {s: d.label.values for s, d in splits.items()}
idx = {s: np.arange(len(v)) for s, v in y.items()}
if SMOKE:   # 16 tweets per class from each split
    idx = {s: np.concatenate([np.where(v == c)[0][:16] for c in range(len(LABELS))]) for s, v in y.items()}
    texts = {s: [texts[s][i] for i in idx[s]] for s in texts}
    y = {s: y[s][idx[s]] for s in y}

## 1. How each tokenizer sees Kiswahili

**Fertility** is the average number of subword tokens per word. A tokenizer that splits Kiswahili words into many tiny pieces has a weaker representation of the language (Rust et al., 2021). Models pretrained on more Kiswahili should have lower fertility.

In [ ]:
sample = texts["train"][:300]
n_words = sum(len(t.split()) for t in sample)
rows = []
for name, ckpt in MODELS.items():
    tok = AutoTokenizer.from_pretrained(ckpt)
    ids = [tok(t, add_special_tokens=False)["input_ids"] for t in sample]
    unk = sum(i == tok.unk_token_id for seq in ids for i in seq)
    rows.append({"model": name, "vocab size": tok.vocab_size, "fertility (subwords/word)": sum(map(len, ids)) / n_words,
                 "% [UNK]": 100 * unk / sum(map(len, ids))})
display(pd.DataFrame(rows).round(2))
tok = AutoTokenizer.from_pretrained(list(MODELS.values())[-1])
print("Example:", sample[3]); print(" ->", tok.tokenize(sample[3]))

## 2. Dataset, weighted loss and training setup

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings, self.labels = encodings, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item["labels"] = int(self.labels[i])
        return item


counts = np.bincount(y["train"], minlength=len(LABELS))
CLASS_WEIGHTS = torch.tensor(len(y["train"]) / (len(LABELS) * counts), dtype=torch.float32)


class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy (the default loss is unweighted)."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss = nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


BASE_CFG = dict(lr=3e-5, batch_size=16, epochs=1 if SMOKE else 5, weight_decay=0.01, warmup_ratio=0.1,
                max_length=128, class_weight=True, freeze_encoder=False)


def fine_tune(checkpoint, cfg, seed):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    data = {s: TweetDataset(tokenizer(texts[s], truncation=True, max_length=cfg["max_length"]), y[s]) for s in texts}
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
    if cfg["freeze_encoder"]:
        for name, p in model.named_parameters():
            if not name.startswith(("classifier", "pre_classifier")):
                p.requires_grad = False

    out_dir = "checkpoints/tmp"
    # warmup given in steps (not warmup_ratio) so this works with transformers v4 and v5
    steps_per_epoch = -(-len(y["train"]) // cfg["batch_size"])
    warmup_steps = int(cfg["warmup_ratio"] * steps_per_epoch * cfg["epochs"])
    args = TrainingArguments(
        output_dir=out_dir, seed=seed, learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch_size"], per_device_eval_batch_size=64,
        num_train_epochs=cfg["epochs"], weight_decay=cfg["weight_decay"], warmup_steps=warmup_steps,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=torch.cuda.is_available(), logging_strategy="epoch", report_to="none",
    )
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=data["train"], eval_dataset=data["validation"],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=lambda p: score(p.label_ids, p.predictions.argmax(-1)),
        class_weights=CLASS_WEIGHTS if cfg["class_weight"] else None,
    )
    trainer.train()
    shutil.rmtree(out_dir, ignore_errors=True)
    val_logits = trainer.predict(data["validation"]).predictions
    test_logits = trainer.predict(data["test"]).predictions
    history = pd.DataFrame([r for r in trainer.state.log_history if "eval_macro_f1" in r])[["epoch", "eval_loss", "eval_macro_f1"]]
    return trainer.model, tokenizer, val_logits, test_logits, history


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


BEST = {"val_f1": -1}   # best single run across all experiments; this one is saved for deployment


def run_experiment(name, checkpoint, **overrides):
    cfg = {**BASE_CFG, **overrides, "checkpoint": checkpoint}
    val_runs, test_runs = [], []
    for seed in SEEDS:
        model, tokenizer, val_logits, test_logits, history = fine_tune(checkpoint, cfg, seed)
        val_m, test_m = score(y["validation"], val_logits.argmax(1)), score(y["test"], test_logits.argmax(1))
        val_runs.append(val_m); test_runs.append(test_m)
        print(f"  {name} | seed {seed}: val macro-F1 {val_m['macro_f1']:.3f} | test macro-F1 {test_m['macro_f1']:.3f}")
        if val_m["macro_f1"] > BEST["val_f1"]:
            BEST.clear()
            BEST.update(name=name, seed=seed, val_f1=val_m["macro_f1"], model=model.cpu(), tokenizer=tokenizer,
                        test_proba=softmax(test_logits), history=history, cfg=cfg)
        else:
            del model
        gc.collect(); torch.cuda.empty_cache()
    val_agg, test_agg = aggregate_seeds(val_runs), aggregate_seeds(test_runs)
    if not SMOKE:
        log_experiment(name, "validation", val_agg, cfg)
        log_experiment(name, "test", test_agg, cfg)
    print(f"{name}: val {val_agg['macro_f1']:.3f} ± {val_agg['macro_f1_std']:.3f} | "
          f"test {test_agg['macro_f1']:.3f} ± {test_agg['macro_f1_std']:.3f}\n")
    return val_agg["macro_f1"]

## 3. T1–T4: comparing pretrained encoders

All four models get identical hyperparameters (lr 3e-5, batch 16, 5 epochs, 10% warmup, weight decay 0.01, weighted loss), so differences come from the pretraining. The best epoch is selected by validation macro-F1.

In [ ]:
val_scores = {}
for name, ckpt in MODELS.items():
    val_scores[name] = run_experiment(name, ckpt)

## 4. T5–T7: ablations on the best encoder

In [ ]:
best_name = max(val_scores, key=val_scores.get)
best_ckpt = MODELS[best_name]
print("Best encoder on validation:", best_name)

if not SMOKE:
    run_experiment(f"T5 {best_name[3:]} without class weights", best_ckpt, class_weight=False)
    # Frozen encoder: only the small head trains, so it needs a much higher learning rate and more epochs
    run_experiment(f"T6 {best_name[3:]} frozen encoder", best_ckpt, freeze_encoder=True, lr=1e-3, epochs=10)
    if RUN_LARGE:
        run_experiment("T7 AfroXLMR large", "Davlan/afro-xlmr-large", lr=1e-5, batch_size=16)

## 5. Results across all notebooks

In [ ]:
log = pd.read_csv("results/experiments.csv")
cols = ["macro_f1", "macro_f1_std", "accuracy", "macro_precision", "macro_recall", "f1_positive", "f1_neutral", "f1_negative"]
test = log[log.split == "test"].set_index("experiment")
val = log[log.split == "validation"].set_index("experiment")["macro_f1"].rename("val_macro_f1")
summary = test.reindex(columns=cols).join(val).sort_values("val_macro_f1", ascending=False)
summary.round(3)

In [ ]:
import matplotlib.pyplot as plt
plot_df = summary.dropna(subset=["macro_f1"]).sort_values("macro_f1")
colors = ["#e76f51" if i.startswith("T") else "#2a9d8f" if i.startswith("L") else "#adb5bd" for i in plot_df.index]
fig, ax = plt.subplots(figsize=(7, 0.35 * len(plot_df) + 1))
ax.barh(plot_df.index, plot_df.macro_f1, xerr=plot_df.macro_f1_std.fillna(0), color=colors)
ax.set_xlabel("Test macro-F1 (± std over 3 seeds)")
ax.set_title("All experiments (test set)")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=l) for c, l in [("#adb5bd", "TF-IDF"), ("#2a9d8f", "BiLSTM"), ("#e76f51", "Transformer")]],
          loc="lower right")
fig.tight_layout(); fig.savefig("results/figures/all_experiments.png", dpi=150); plt.show()

## 6. Best model: per-class results and confusion matrix

In [ ]:
print(f"Best single run: {BEST['name']} (seed {BEST['seed']}), val macro-F1 {BEST['val_f1']:.3f}")
test_pred = BEST["test_proba"].argmax(1)
print(report(y["test"], test_pred))
_ = plot_confusion(y["test"], test_pred, f"{BEST['name']} (test)", save_as="cm_best_transformer.png")
BEST["history"]

## 7. Save the model and its predictions

The saved folder (`models/best_transformer`) is what the web app loads. Model weights are too large for GitHub, so section 8 uploads them to the **Hugging Face Hub**, and the deployed app downloads them from there.

In [ ]:
os.makedirs("models", exist_ok=True)
BEST["model"].save_pretrained("models/best_transformer")
BEST["tokenizer"].save_pretrained("models/best_transformer")
preds = splits["test"].iloc[idx["test"]][["text", "label_name"]].assign(
    pred=[LABELS[i] for i in test_pred],
    **{f"p_{lab}": BEST["test_proba"][:, i].round(4) for i, lab in enumerate(LABELS)})
if not SMOKE:
    preds.to_csv("results/predictions_transformer.csv", index=False)
preds.head()

## 8. Upload the model and results to Hugging Face

This creates **`<you>/kiswahili-sentiment`** on your Hugging Face account. It holds the fine-tuned model, a model card, and a `results/` folder (the experiment table, predictions and figures, which get copied back into the GitHub repo).

The web app (`app/streamlit_app.py`) is deployed separately on Streamlit Community Cloud and downloads this model. Hugging Face hosting of Gradio Spaces now requires a paid PRO account.

In [ ]:
if UPLOAD_TO_HUB:
    from huggingface_hub import HfApi
    api = HfApi()
    MODEL_REPO = f"{HF_USER}/kiswahili-sentiment"
    m = score(y["test"], test_pred)

    # 1) model + tokenizer
    api.create_repo(MODEL_REPO, exist_ok=True)
    api.upload_folder(folder_path="models/best_transformer", repo_id=MODEL_REPO,
                      commit_message=f"Upload {BEST['name']} (seed {BEST['seed']})")

    # 2) model card
    card = f"""---
language: sw
pipeline_tag: text-classification
base_model: {BEST['cfg']['checkpoint']}
datasets:
- shmuhammad/AfriSenti-twitter-sentiment
metrics:
- f1
---

# Kiswahili tweet sentiment ({BEST['name'][3:]})

`{BEST['cfg']['checkpoint']}` fine-tuned on the Kiswahili subset of
[AfriSenti](https://huggingface.co/datasets/shmuhammad/AfriSenti-twitter-sentiment) (Muhammad et al., 2023)
to classify tweets as **positive / neutral / negative**.

| Test metric | Value |
|---|---|
| Macro-F1 | {m['macro_f1']:.3f} |
| Accuracy | {m['accuracy']:.3f} |
| F1 positive / neutral / negative | {m['f1_positive']:.3f} / {m['f1_neutral']:.3f} / {m['f1_negative']:.3f} |

Training: lr {BEST['cfg']['lr']}, batch {BEST['cfg']['batch_size']}, {BEST['cfg']['epochs']} epochs, {'class-weighted' if BEST['cfg']['class_weight'] else 'unweighted'} cross-entropy,
best epoch selected on validation macro-F1 (seed {BEST['seed']}).

Code, experiments and error analysis: https://github.com/Joellate/kiswahili-sentiment-analysis
Demo: https://kiswahili-sentiment.streamlit.app

**Limitations:** trained on only 1,810 tweets; emojis and punctuation were removed from the training data;
struggles with sarcasm, slang/Sheng and with negative *news* vs negative *opinion*.
"""
    api.upload_file(path_or_fileobj=card.encode("utf-8"), path_in_repo="README.md", repo_id=MODEL_REPO,
                    commit_message="Add model card")

    # 3) results (copied back into the GitHub repo afterwards)
    api.upload_folder(folder_path="results", path_in_repo="results", repo_id=MODEL_REPO,
                      commit_message="Upload experiment results")

    print("Model:   https://huggingface.co/" + MODEL_REPO)
    print("Results: https://huggingface.co/" + MODEL_REPO + "/tree/main/results")
    print("Web app: deploy app/streamlit_app.py on Streamlit Community Cloud (see README).")
    print("         It downloads the model above, so update MODEL_ID in app/predict.py if your username differs.")

## Takeaways (results of the Colab T4 run; mean ± std over 3 seeds, test set)

| | Test macro-F1 | F1 negative | Tokenizer fertility |
|---|---|---|---|
| T1 mBERT | 0.432 ± 0.020 | 0.26 | 2.12 |
| T2 XLM-R base | 0.388 ± 0.149 | 0.29 | 1.61 |
| T3 AfriBERTa large | 0.556 ± 0.013 | 0.52 | **1.45** |
| T4 AfroXLMR base | 0.554 ± 0.009 | 0.53 | 1.61 |
| T5 AfriBERTa without class weights | **0.566 ± 0.005** | 0.53 | |
| T6 AfriBERTa, frozen encoder | 0.537 ± 0.007 | 0.51 | |
| *Best TF-IDF / best BiLSTM* | *0.486 / 0.455* | *0.31 / 0.31* | |

Fertility = subword tokens per word, measured on all 1,810 training tweets.

1. **Pretraining on African languages is what makes the difference.** AfriBERTa and AfroXLMR reach about 0.555 macro-F1, roughly 0.07 above the best TF-IDF model (0.486) and 0.10 above the BiLSTM. Negative-class F1 also jumps from about 0.31 to about 0.52. Notebook 05 shows that this gain is statistically significant: the paired-bootstrap 95% CI is [+0.065, +0.161]. The models bring Kiswahili knowledge that 1,810 tweets cannot supply.
2. **General multilingual models do not help at all.** mBERT (0.432) and XLM-R (0.388) score *below* the TF-IDF baseline. Kiswahili is only a tiny share of their pretraining data. The cleanest comparison is **XLM-R vs AfroXLMR**: they have the same architecture and the *same tokenizer*. The only difference is that AfroXLMR continued pretraining on African-language text (MAFT; Alabi et al., 2022), and that gives +0.17 macro-F1.
3. **Tokenization matters, but it is not the whole story.** AfriBERTa's tokenizer splits Kiswahili least (1.45 pieces per word). It keeps *wamefariki* ("they have died") as one token, whereas mBERT cuts it into `wa ##me ##fari ##ki` (2.12 pieces per word). But XLM-R and AfroXLMR have identical fertility (1.61) and very different scores, so **what the model learned during pretraining** matters more than how it splits words. AfriBERTa matches the 2× larger AfroXLMR despite being pretrained on far less text, consistent with Ogueji et al. (2021).
4. **XLM-R is unstable on small data.** Its std of 0.149 means at least one of the three seeds failed to train properly (a mean of 0.388 with that spread implies one run far below the others). Fine-tuning large Transformers on small datasets is known to be seed-sensitive (Mosbach et al., 2021). This is why every result here is averaged over 3 seeds.
5. **Class weighting is no longer needed.** T5 (0.566 ± 0.005) vs T3 (0.556 ± 0.013) is within noise, and negative-class F1 is the same. For TF-IDF, removing weights cost 0.08 macro-F1. The pretrained representations already separate the negative class well, so the classifier does not need the loss to push it towards that class.
6. **Most of the gain comes from the pretrained representations themselves.** With the encoder **frozen**, so that only the small classification head is trained, AfriBERTa still reaches 0.537 and beats every non-Transformer model. Full fine-tuning adds a further ~0.02–0.03.
7. **What is left is the neutral boundary.** Per notebook 05, 97% of the best model's errors involve *neutral*. Positive ↔ negative flips are almost nonexistent (3.5% of errors).

**Model deployed:** T5 (AfriBERTa-large, unweighted loss), the best single run on validation (seed 42), with 0.568 test macro-F1.

In [ ]:
# Fallback only if the Hugging Face upload was skipped: download the results folder instead
if "google.colab" in sys.modules and not UPLOAD_TO_HUB:
    !zip -qr results_04.zip results
    from google.colab import files
    files.download("results_04.zip")